In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

In [ ]:
import re
import string
import joblib
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

from wordcloud import WordCloud

import nltk
from nltk.probability import FreqDist

import gensim
from gensim import corpora
from gensim.models import Phrases
from gensim.models.phrases import Phraser

In [ ]:
import spacy
nlp = spacy.load("uk_core_news_sm")
import uk_core_news_sm
nlp = uk_core_news_sm.load()

### Preprocessing data (adding columns `clean_text` and `label`)

In [ ]:
# conn = sqlite3.connect('../data/news.db')
# query = '''
#         SELECT * 
#         FROM articles
#         JOIN preds ON articles.id = preds.article_id
#         WHERE articles.source_id = 2
#         '''
# df = pd.read_sql_query(query, conn)
# conn.close()

In [ ]:
# df = df[(df['date'] >= '2024-04-01 00:00:00') & (df['date'] < '2024-05-01 00:00:00')]

In [ ]:
# def preprocess(text):
#     text = text.lower()
#     text = re.sub('\[.*?\]', '', text)
#     text = re.sub("\\W"," ",text) 
#     text = re.sub('https?://\S+|www\.\S+', '', text)
#     text = re.sub('<.*?>+', '', text)
#     text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
#     text = re.sub('\n', '', text)
#     text = re.sub('\w*\d\w*', '', text)
    
#     doc = nlp(text)
#     lemmatized_tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]
    
#     return ' '.join(lemmatized_tokens)

In [ ]:
# seqs_to_del = [
#     'підписуйтесь канал telegram viber',
#     'термінові важливий',
#     'терміновий важливий',
#     'повідомлення війна росія україна читати канал рбк україна telegram',
#     'повідомляти рбк україна ',
#     'рбк україна ',
#     'читати',
    
#     'підписуватися ukraine now', 
#     'babel', 
#     'bloomberg',
#     'spravdi',
#     'informnapalm',
#     'підписатись',
#     'сайт',
#     'нин', 
#     'https'
#     'telegram', 
#     'instagram', 
#     'twitter', 
#     'facebook', 
#     'viber',
    
#     'україна', 
#     'український',
#     'росія',
#     'російський',
# ]

# def preprocess(text):
#     text = text.lower()
#     text = re.sub('\[.*?\]', '', text)
#     text = re.sub("\\W"," ",text) 
#     text = re.sub('https?://\S+|www\.\S+', '', text)
#     text = re.sub('<.*?>+', '', text)
#     text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
#     text = re.sub('\n', '', text)
#     text = re.sub('\w*\d\w*', '', text)
    
#     doc = nlp(text)
#     lemmatized_tokens = \
#         [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]
#     text = ' '.join(lemmatized_tokens)

#     for seq in seqs_to_del:
#         text = text.replace(seq, '')
#     text = re.sub(r'\s+', ' ', text)
#     text = text.strip()

#     return text

In [ ]:
# df['clean_text'] = df['text'].apply(preprocess)

In [ ]:
# class TextPreprocessor(BaseEstimator, TransformerMixin):
#     def __init__(self):
#         pass
    
#     def fit(self, X, y=None):
#         return self
    
#     def transform(self, X):
#         preprocessed_texts = []
#         for text in X:
#             preprocessed_text = self.preprocess_text(text)
#             preprocessed_texts.append(preprocessed_text)
#         return preprocessed_texts
    
#     def preprocess_text(self, text):
#         text = text.lower()
#         text = re.sub('\[.*?\]', '', text)
#         text = re.sub("\\W"," ",text) 
#         text = re.sub('https?://\S+|www\.\S+', '', text)
#         text = re.sub('<.*?>+', '', text)
#         text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
#         text = re.sub('\n', '', text)
#         text = re.sub('\w*\d\w*', '', text)
#         return text

In [ ]:
def preprocess_title(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text) 
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [ ]:
df['clean_title'] = df['title'].apply(preprocess_title)

In [ ]:
model = joblib.load('../models/model_pipeline_1715225089.pkl')

In [ ]:
preds = model.predict_proba(df['clean_title'])

In [ ]:
df['label'] = preds[:,1:]

In [ ]:
# df.to_csv('data2024april.csv', index=False)
# df = pd.read_csv('../data/data2024april.csv')

In [ ]:
df[df['label'] < 0.5]

### Inspecting words in dataset

In [ ]:
# Пример данных
data = {
    'название': ['Статья 1', 'Статья 2', 'Статья 3', 'Статья 4', 'Статья 5'],
    'count': [100, 50, 200, 75, 150]
}

# Создание DataFrame
df = pd.DataFrame(data)

# Получение 5 записей с наибольшими значениями в столбце 'count'
top_5_records = df.nlargest(5, 'count')

top_5_records

In [ ]:
wc = WordCloud(background_color="black", 
               max_words=100,
               max_font_size=256,
               random_state=0, 
               width=1000, 
               height=1000)
wc.generate(' '.join(df['clean_text']))

plt.figure(figsize=(10, 5))
plt.axis('off')
plt.imshow(wc, interpolation="bilinear")
# plt.savefig('./123.png')
# plt.show()

In [ ]:
text_tokens = [txt.split() for txt in df['clean_text']]
texts = ' '.join(df['clean_text'])
texts = texts.split()

In [ ]:
freq_dist = FreqDist(texts)
keywords = [word for word, _ in freq_dist.most_common(10)]

print("Частотные ключевые слова:", keywords)

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_text'])
    
feature_names = vectorizer.get_feature_names_out()
tfidf_scores = X.toarray()[0]
keyword_indices = tfidf_scores.argsort()[-10:][::-1]
keywords_tfidf = [feature_names[i] for i in keyword_indices]
    
print("Ключевые слова по TF-IDF:", keywords_tfidf)

In [ ]:
all_words = []
sub_df = df[df['source_id'] == 11]
for text in sub_df['clean_text']:
    words = text.split()
    all_words.extend(words)

word_counts = Counter(all_words)

length = len(df) * 0.01
common_words = [word for word, count in word_counts.items() if count >= length]

print("Общие слова, которые встречаются в каждом тексте:")
print(common_words)

### Topic modeling

In [ ]:
text_tokens = [txt.split() for txt in df['clean_text']]

In [ ]:
bigram = Phrases(text_tokens, min_count=5, threshold=10)
bigram_phraser = Phraser(bigram)

bigram_text = [bigram_phraser[txt] for txt in text_tokens]

In [ ]:
dictionary = corpora.Dictionary(bigram_text)
corpus = [dictionary.doc2bow(txt) for txt in bigram_text]

In [ ]:
num_topics = 5
lda_model = gensim.models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics)

In [ ]:
lda_model.print_topics()
# lda_model.show_topics(formatted=False)

In [ ]:
for idx, topic in lda_model.print_topics(num_words=5):
    print('Topic: {}'.format(idx))
    words = topic.split('"')[1::2]
    print('Words:', words)

In [ ]:
# # Generate word clouds for each topic
# for idx, topic in lda_model.show_topics(formatted=False):
#     word_freq = {word: freq for word, freq in topic}
#     wordcloud = WordCloud(background_color='white').generate_from_frequencies(word_freq)
    
#     # Plot word cloud
#     plt.figure(figsize=(8, 6))
#     plt.imshow(wordcloud, interpolation='bilinear')
#     plt.title('Topic {}'.format(idx))
#     plt.axis('off')
#     plt.show()

In [ ]:
# # Plot word frequencies for each topic
# for idx, topic in lda_model.show_topics(formatted=False):
#     word_freq = {word: freq for word, freq in topic}
#     sorted_word_freq = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
#     top_words = sorted_word_freq[:10]  # Select top 10 words
#     words, freqs = zip(*top_words)
    
#     # Plot bar plot
#     plt.figure(figsize=(8, 6))
#     plt.barh(words, freqs, color='skyblue')
#     plt.gca().invert_yaxis()
#     plt.xlabel('Word Frequency')
#     plt.title('Topic {}'.format(idx))
#     plt.show()

In [ ]:
# # Подсчет количества статей для каждой темы
# topic_counts = [0] * num_topics  # Создаем список нулей для каждой темы
# for doc_topics in lda_model.get_document_topics(corpus):
#     for topic, _ in doc_topics:
#         topic_counts[topic] += 1

# # Визуализация количества статей для каждой темы
# plt.figure(figsize=(10, 6))
# plt.bar(range(num_topics), topic_counts, color='skyblue')
# plt.xlabel('Topic')
# plt.ylabel('Number of Articles')
# plt.title('Number of Articles per Topic')
# plt.xticks(range(num_topics))
# plt.show()

### Cluster analysis

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
matrix = vectorizer.fit_transform(df['clean_text'])

In [ ]:
kmeans = KMeans(n_clusters=4)
kmeans.fit(matrix)

In [ ]:
_, topic_counts = np.unique(kmeans.labels_, return_counts=True)

In [ ]:
feature_names = vectorizer.get_feature_names_out()
top_keywords = []

for cluster_center in kmeans.cluster_centers_:
    top_keyword_idxs = cluster_center.argsort()[-5:][::-1]
    top_keywords.append([feature_names[idx] for idx in top_keyword_idxs])

In [ ]:
top_keywords

In [ ]:
print("Наиболее обсуждаемые темы и их ключевые слова:")
for i, count in enumerate(topic_counts):
    print(f"Тема {i}: {count} статей")
    print("Ключевые слова:", ', '.join(top_keywords[i]))
    print()

In [ ]:
pca = PCA(n_components=2)
reduced_features = pca.fit_transform(matrix.toarray())

plt.figure(figsize=(10, 7))
sns.scatterplot(x=reduced_features[:, 0], 
                y=reduced_features[:, 1], 
                hue=kmeans.labels_, 
                palette='deep', 
                legend='full')
plt.title('Clusters of News Articles')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.grid(True)
plt.show()

In [ ]:
df['kmean_label'] = kmeans.labels_

In [ ]:
# topics = list(df['kmean_label'].value_counts().index)
# counts = list(df['kmean_label'].value_counts())
# fake_counts_values = [len(df[(df['kmean_label'] == topic) & (df['label'] < 0.5)]) for topic in topics]

# plt.figure(figsize=(6, 4))
# plt.bar(topics, counts, color='#87cefa', label='Всего статей')
# plt.bar(topics, fake_counts_values, color='#ffc87c', label='Фейковые статьи')

# plt.title('Результаты кластерного анализа')
# plt.xlabel('Темы')
# plt.ylabel('Количество статей')
# plt.legend()
# plt.show()

In [ ]:
topics = list(df['kmean_label'].value_counts().index)
counts = list(df['kmean_label'].value_counts())
fake_counts_values = [len(df[(df['kmean_label'] == topic) & (df['label'] < 0.5)]) for topic in topics]

df_sns = pd.DataFrame({
    'topic': topics,
    'total count': counts,
    'fakes count': fake_counts_values
})
df_long = df_sns.melt(id_vars='topic', var_name='type', value_name='count')

plt.figure(figsize=(6, 4))
sns.barplot(x='topic', y='count', hue='type', data=df_long, palette=['#87cefa', '#ffc87c'])

plt.title('Articles by topics')
plt.xlabel('Topics')
plt.ylabel('Articles count')
# plt.savefig('my_plot.png')
plt.show()

In [ ]:
sources = list(df['source_id'].value_counts().index)
counts = list(df['source_id'].value_counts())
fake_counts_values = [len(df[(df['source_id'] == source) & (df['label'] < 0.5)]) for source in sources]

df_sns = pd.DataFrame({
    'source': sources,
    'total count': counts,
    'fakes count': fake_counts_values
})
df_long = df_sns.melt(id_vars='source', var_name='type', value_name='count')

plt.figure(figsize=(6, 4))
sns.barplot(x='source', y='count', hue='type', data=df_long, palette=['#87cefa', '#ffc87c'])

plt.title('Articles by sources')
plt.xlabel('Sources')
plt.ylabel('Articles count')
# plt.savefig('my_plot_2.png')
plt.show()

### Sentiment analysis

In [ ]:
from textblob import TextBlob

# Пример текста для анализа
text = "thw movie was terribly good"

# Создание объекта TextBlob
blob = TextBlob(text)

# Анализ тональности текста
sentiment_score = blob.sentiment.polarity

# Определение эмоциональной окраски
if sentiment_score > 0:
    print("Текст положительный.")
elif sentiment_score < 0:
    print("Текст отрицательный.")
else:
    print("Текст нейтральный.")

### Similarities calculation

In [ ]:
def find_similar_texts(sentence, texts):
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([sentence] + texts)
    
    similarity_scores = cosine_similarity(vectors[0], vectors[1:])[0]
    
    ranked_texts = [(text, score) for text, score in zip(texts, similarity_scores)]
    ranked_texts.sort(key=lambda x: x[1], reverse=True)
    
    return ranked_texts

In [ ]:
topic = preprocessor('')

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([topic] + list(df['clean_text']))

scores = cosine_similarity(vectors[0], vectors[1:])[0]

df['score'] = scores
df = df.sort_values(by='score', ascending=False)

In [ ]:
df